# Visual Abstract: State-Aware IDP Ensemble Modeling

This notebook generates all 5 panels for the visual abstract and composites them into a single publication-ready figure.

**Layout:** 3-column, top-to-bottom reading flow: Problem → Method → Outcome
- **Row 1** (Problem): Panel 1 + Panel 2
- **Row 2** (Method): Panel 3 + Panel 4
- **Row 3** (Outcome): Panel 5 (spanning)

All panels are generated with matplotlib — no external molecular viewers required.

In [ ]:
#@title **Cell 1 — Install Dependencies**
!pip install -q matplotlib numpy scipy Pillow

In [ ]:
#@title **Cell 2 — Imports & Color Palette**
import numpy as np
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, Ellipse, Polygon, Circle
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image, ImageDraw, ImageFont
import glob, os, warnings
warnings.filterwarnings('ignore')

# === CONSISTENT COLOR PALETTE ===
PAL = {
    'bg':          '#FFFFFF',
    'text':        '#1a1a2e',
    'accent_blue': '#2563EB',
    'accent_red':  '#DC2626',
    'orange':      '#F59E0B',
    'grey':        '#9CA3AF',
    'light_grey':  '#E5E7EB',
    'magenta':     '#D946EF',
    'yellow':      '#FBBF24',
    'cyan':        '#06B6D4',
    'green':       '#10B981',
    'purple':      '#7C3AED',
    'purple_bg':   '#EDE9FE',
    'purple_dark': '#5B21B6',
    'dark_blue':   '#1E3A5F',
    'light_bg':    '#F0F4FF',
    'pink_bg':     '#FCE7F3',
    'brown':       '#92400E',
}

DPI = 300
os.makedirs('panels', exist_ok=True)
print('Setup complete. Panels will be saved to panels/ directory.')

In [ ]:
#@title **Cell 3 — Generate Synthetic IDP Ensemble**
#
# This creates ~30 IDP-like backbone conformers using a persistent random walk.
# Replace this with your own MD/NMR ensemble coordinates if you have them.

def generate_idp_ensemble(n_conformers=30, n_residues=80, seed=42):
    """Generate synthetic IDP backbone conformers (random walk with persistence)."""
    rng = np.random.default_rng(seed)
    conformers = []
    for _ in range(n_conformers):
        coords = np.zeros((n_residues, 3))
        d = rng.standard_normal(3)
        d /= np.linalg.norm(d)
        for j in range(1, n_residues):
            step = d + 0.6 * rng.standard_normal(3)
            step = 3.8 * step / np.linalg.norm(step)  # ~3.8 A CA-CA distance
            coords[j] = coords[j - 1] + step
            d = step / np.linalg.norm(step)
        coords -= coords.mean(axis=0)  # center
        conformers.append(coords)
    return conformers

ensemble = generate_idp_ensemble(n_conformers=30, n_residues=80)
print(f'Generated {len(ensemble)} conformers, {ensemble[0].shape[0]} residues each.')

In [ ]:
#@title **Cell 4 — Panel 1: IDP Ensemble Reality**
#
# Goal: Show that IDPs are flexible ensembles, not single folds.
# Overlaid conformers colored blue→red with semi-transparency.

fig = plt.figure(figsize=(8, 7), facecolor=PAL['bg'])
ax = fig.add_subplot(111, projection='3d', facecolor=PAL['bg'])

cmap = LinearSegmentedColormap.from_list('br', [PAL['accent_blue'], PAL['accent_red']])
n_res = ensemble[0].shape[0]

for conf in ensemble:
    t = np.linspace(0, 1, n_res)
    t_fine = np.linspace(0, 1, 300)
    smooth = np.column_stack([np.interp(t_fine, t, conf[:, k]) for k in range(3)])
    colors = cmap(t_fine)
    colors[:, 3] = 0.25  # semi-transparent
    for i in range(len(smooth) - 1):
        ax.plot(smooth[i:i+2, 0], smooth[i:i+2, 1], smooth[i:i+2, 2],
                color=colors[i], linewidth=1.2)

ax.text2D(0.5, 0.96, 'BIOLOGICAL PROBLEM', transform=ax.transAxes, ha='center',
          fontsize=9, color=PAL['grey'], fontweight='bold')
ax.set_title('IDP Ensemble Reality', fontsize=16, fontweight='bold',
             color=PAL['text'], pad=25)
ax.text2D(0.5, 0.02,
          'IDPs exist as flexible ensembles, not a single static structure.',
          transform=ax.transAxes, ha='center', fontsize=10,
          color=PAL['text'], style='italic')

ax.set_axis_off()
ax.view_init(elev=20, azim=45)

plt.tight_layout()
plt.savefig('panels/panel1_idp_ensemble.png', dpi=DPI, bbox_inches='tight',
            facecolor=PAL['bg'])
plt.show()
print('Panel 1 saved → panels/panel1_idp_ensemble.png')

In [ ]:
#@title **Cell 5 — Panel 2: Static Structure Failure**
#
# Goal: Visual contrast — one rigid structure vs. ensemble cloud with hidden pockets.
# Left: single ribbon. Right: ensemble + transient pocket surfaces (yellow spheres).

fig = plt.figure(figsize=(14, 6), facecolor=PAL['bg'])

# --- Left subpanel: Single static structure ---
ax1 = fig.add_subplot(121, projection='3d', facecolor=PAL['bg'])
single = ensemble[0]
t = np.linspace(0, 1, single.shape[0])
t_fine = np.linspace(0, 1, 300)
smooth = np.column_stack([np.interp(t_fine, t, single[:, k]) for k in range(3)])
ax1.plot(smooth[:, 0], smooth[:, 1], smooth[:, 2],
         color=PAL['accent_blue'], linewidth=2.5, alpha=0.95)
ax1.set_title('Static docking target', fontsize=13, fontweight='bold',
              color=PAL['text'])
ax1.set_axis_off()
ax1.view_init(elev=20, azim=45)

# --- Right subpanel: Ensemble + transient pockets ---
ax2 = fig.add_subplot(122, projection='3d', facecolor=PAL['bg'])
cmap_br = LinearSegmentedColormap.from_list('br', [PAL['accent_blue'], PAL['accent_red']])
for conf in ensemble:
    t = np.linspace(0, 1, conf.shape[0])
    t_fine = np.linspace(0, 1, 300)
    sm = np.column_stack([np.interp(t_fine, t, conf[:, k]) for k in range(3)])
    colors = cmap_br(t_fine)
    colors[:, 3] = 0.15
    for i in range(len(sm) - 1):
        ax2.plot(sm[i:i+2, 0], sm[i:i+2, 1], sm[i:i+2, 2],
                 color=colors[i], linewidth=0.8)

# Transient pocket spheres (yellow)
rng = np.random.default_rng(99)
pocket_centers = [ensemble[j][rng.integers(20, 60)] for j in [3, 10, 22]]
u, v = np.mgrid[0:2*np.pi:20j, 0:np.pi:10j]
for pc in pocket_centers:
    r = 5.0
    xs = pc[0] + r * np.cos(u) * np.sin(v)
    ys = pc[1] + r * np.sin(u) * np.sin(v)
    zs = pc[2] + r * np.cos(v)
    ax2.plot_surface(xs, ys, zs, color=PAL['yellow'], alpha=0.35)

ax2.set_title('Dynamic ensemble with\ntransient cryptic pockets', fontsize=13,
              fontweight='bold', color=PAL['text'])
ax2.set_axis_off()
ax2.view_init(elev=20, azim=45)

fig.suptitle('STATIC STRUCTURE FAILURE', fontsize=16, fontweight='bold',
             y=1.02, color=PAL['text'])
fig.text(0.5, -0.02,
         'Static docking on one PDB misses transient, cryptic binding pockets.',
         ha='center', fontsize=11, color=PAL['text'], style='italic')

plt.tight_layout()
plt.savefig('panels/panel2_static_vs_dynamic.png', dpi=DPI, bbox_inches='tight',
            facecolor=PAL['bg'])
plt.show()
print('Panel 2 saved → panels/panel2_static_vs_dynamic.png')

In [ ]:
#@title **Cell 6 — Panel 3: Pocket-State Modeling (tICA Scatter)**
#
# Goal: Show clustering into OPEN / CLOSED pocket macrostates.
# Left: tICA scatter plot with clusters circled.
# Right: Representative structures for each macrostate.

fig = plt.figure(figsize=(14, 6), facecolor=PAL['bg'])

# ---- Left: tICA scatter ----
ax_sc = fig.add_subplot(121, facecolor=PAL['light_bg'])

rng = np.random.default_rng(7)
# Cluster 1: OPEN
n1 = 400
c1_x = rng.normal(-2.5, 0.8, n1)
c1_y = rng.normal(1.5, 0.7, n1)
# Cluster 2: CLOSED
n2 = 350
c2_x = rng.normal(2.0, 0.9, n2)
c2_y = rng.normal(-1.5, 0.8, n2)
# Ambiguous
n3 = 150
c3_x = rng.normal(0.0, 1.2, n3)
c3_y = rng.normal(0.0, 1.0, n3)

ax_sc.scatter(c3_x, c3_y, c=PAL['grey'], alpha=0.3, s=12, label='Ambiguous')
ax_sc.scatter(c1_x, c1_y, c=PAL['orange'], alpha=0.55, s=15, label='OPEN pocket')
ax_sc.scatter(c2_x, c2_y, c=PAL['accent_blue'], alpha=0.55, s=15, label='CLOSED pocket')

# Dashed ellipses around clusters
ell1 = Ellipse((-2.5, 1.5), 4.0, 3.5, fill=False,
               edgecolor=PAL['orange'], linewidth=2, linestyle='--')
ell2 = Ellipse((2.0, -1.5), 4.5, 3.8, fill=False,
               edgecolor=PAL['accent_blue'], linewidth=2, linestyle='--')
ax_sc.add_patch(ell1)
ax_sc.add_patch(ell2)

ax_sc.set_xlabel('tIC 1', fontsize=12, color=PAL['text'])
ax_sc.set_ylabel('tIC 2', fontsize=12, color=PAL['text'])
ax_sc.legend(fontsize=10, loc='upper right', framealpha=0.9)
ax_sc.set_title('Conformational Landscape', fontsize=13, fontweight='bold',
                color=PAL['text'])

# ---- Right: Representative structures ----
ax_st = fig.add_subplot(122, facecolor=PAL['bg'])

# OPEN state
open_conf = ensemble[5]
ax_st.plot(open_conf[:, 0], open_conf[:, 1],
           color=PAL['orange'], linewidth=2.2, alpha=0.85)
pocket_idx = 35
circle_open = Circle((open_conf[pocket_idx, 0], open_conf[pocket_idx, 1]),
                     6, color=PAL['yellow'], alpha=0.4, linewidth=2,
                     edgecolor=PAL['orange'], linestyle='--')
ax_st.add_patch(circle_open)
ax_st.text(open_conf[pocket_idx, 0], open_conf[pocket_idx, 1] + 9,
           'OPEN state\nmacrostate', ha='center', fontsize=11,
           fontweight='bold', color=PAL['orange'])

# CLOSED state (shifted down)
closed_conf = ensemble[15]
offset_y = -50
ax_st.plot(closed_conf[:, 0], closed_conf[:, 1] + offset_y,
           color=PAL['accent_blue'], linewidth=2.2, alpha=0.85)
ax_st.text(closed_conf[40, 0], closed_conf[40, 1] + offset_y - 12,
           'CLOSED state\nmacrostate', ha='center', fontsize=11,
           fontweight='bold', color=PAL['accent_blue'])

ax_st.set_axis_off()
ax_st.set_title('Representative Structures', fontsize=13, fontweight='bold',
                color=PAL['text'])

fig.text(0.5, -0.02,
         'We cluster the ensemble into pocket-centric macrostates (OPEN vs CLOSED).',
         ha='center', fontsize=11, color=PAL['text'], style='italic')

plt.tight_layout()
plt.savefig('panels/panel3_pocket_states.png', dpi=DPI, bbox_inches='tight',
            facecolor=PAL['bg'])
plt.show()
print('Panel 3 saved → panels/panel3_pocket_states.png')

In [ ]:
#@title **Cell 7 — Panel 4: SE(3)-Equivariant Diffusion Schematic**
#
# Goal: Three-stage left→right cartoon: Noisy → Model → Clean OPEN-state ensemble.
# Built entirely with matplotlib vector shapes.

fig, ax = plt.subplots(figsize=(16, 5), facecolor=PAL['bg'])
ax.set_xlim(0, 16)
ax.set_ylim(0, 5)
ax.set_axis_off()
ax.set_aspect('equal')

rng = np.random.default_rng(42)

# ===== STAGE 1: Noisy structures =====
for i in range(4):
    x0 = 0.5 + i * 0.9
    y0 = 1.8 + rng.uniform(-0.4, 0.4)
    xs = x0 + np.cumsum(rng.uniform(0.05, 0.15, 12))
    ys = y0 + np.cumsum(rng.normal(0, 0.18, 12))
    ax.plot(xs, ys, color=PAL['grey'], linewidth=1.5, alpha=0.6)
    # noise dots
    for _ in range(8):
        ax.plot(rng.choice(xs) + rng.normal(0, 0.12),
                rng.choice(ys) + rng.normal(0, 0.12),
                'o', color=PAL['grey'], markersize=2, alpha=0.3)

noise_box = FancyBboxPatch((0.2, 0.3), 4.0, 0.7,
                           boxstyle='round,pad=0.15',
                           facecolor=PAL['light_grey'],
                           edgecolor=PAL['grey'], linewidth=1.5)
ax.add_patch(noise_box)
ax.text(2.2, 0.65, 'Noisy structures (t)', ha='center', va='center',
        fontsize=11, color=PAL['text'], fontweight='bold')

# ===== ARROW 1 =====
ax.annotate('', xy=(5.5, 2.5), xytext=(4.5, 2.5),
            arrowprops=dict(arrowstyle='->', color=PAL['text'], lw=2.5))

# ===== STAGE 2: SE(3) diffusion model box =====
model_box = FancyBboxPatch((5.5, 0.8), 5.0, 3.4,
                           boxstyle='round,pad=0.3',
                           facecolor=PAL['purple_bg'],
                           edgecolor=PAL['purple'], linewidth=2.5)
ax.add_patch(model_box)
ax.text(8.0, 3.5, 'SE(3)-Equivariant\nDiffusion Model', ha='center', va='center',
        fontsize=13, fontweight='bold', color=PAL['purple_dark'])

# Circular rotation arrows
angles = np.linspace(0, 1.7 * np.pi, 50)
r = 0.6
cx, cy = 8.0, 1.8
ax.plot(cx + r * np.cos(angles), cy + r * np.sin(angles),
        color=PAL['purple'], linewidth=2, alpha=0.7)
ax.annotate('', xy=(cx + r * np.cos(angles[-1]), cy + r * np.sin(angles[-1])),
            xytext=(cx + r * np.cos(angles[-3]), cy + r * np.sin(angles[-3])),
            arrowprops=dict(arrowstyle='->', color=PAL['purple'], lw=2))

# Mini backbone icon
mini_x = np.array([7.3, 7.6, 7.9, 8.2, 8.5, 8.7])
mini_y = np.array([1.8, 2.05, 1.7, 1.95, 1.65, 1.85])
ax.plot(mini_x, mini_y, color=PAL['purple'], linewidth=2.5, alpha=0.8)

# Conditioning label
cond_box = FancyBboxPatch((5.8, 0.1), 4.4, 0.55,
                          boxstyle='round,pad=0.1',
                          facecolor=PAL['orange'], edgecolor=PAL['orange'],
                          alpha=0.2, linewidth=1.5)
ax.add_patch(cond_box)
ax.text(8.0, 0.37, 'Conditioned on OPEN macrostate', ha='center', va='center',
        fontsize=9, fontweight='bold', color=PAL['brown'])

# ===== ARROW 2 =====
ax.annotate('', xy=(11.8, 2.5), xytext=(10.8, 2.5),
            arrowprops=dict(arrowstyle='->', color=PAL['text'], lw=2.5))

# ===== STAGE 3: Generated OPEN-state ensemble =====
for i in range(3):
    x0 = 12.2 + i * 0.6
    y0 = 1.5 + rng.uniform(-0.2, 0.2)
    xs = x0 + np.cumsum(rng.uniform(0.05, 0.12, 15))
    ys = y0 + np.cumsum(rng.normal(0, 0.08, 15))
    ax.plot(xs, ys, color=PAL['magenta'], linewidth=2, alpha=0.7)

pocket_circle = Circle((13.8, 2.2), 0.5, color=PAL['magenta'],
                        alpha=0.2, linewidth=2,
                        edgecolor=PAL['magenta'], linestyle='--')
ax.add_patch(pocket_circle)

out_box = FancyBboxPatch((11.8, 0.3), 4.0, 0.7,
                         boxstyle='round,pad=0.15',
                         facecolor=PAL['pink_bg'],
                         edgecolor=PAL['magenta'], linewidth=1.5)
ax.add_patch(out_box)
ax.text(13.8, 0.65, 'Generated OPEN-state\nensemble', ha='center', va='center',
        fontsize=10, color=PAL['text'], fontweight='bold')

ax.set_title('SE(3)-EQUIVARIANT DIFFUSION', fontsize=16,
             fontweight='bold', color=PAL['text'], pad=15)
fig.text(0.5, -0.04,
         'State-aware SE(3) diffusion generates 3D structures '
         'conditioned on the OPEN pocket state.',
         ha='center', fontsize=11, color=PAL['text'], style='italic')

plt.savefig('panels/panel4_diffusion.png', dpi=DPI, bbox_inches='tight',
            facecolor=PAL['bg'])
plt.show()
print('Panel 4 saved → panels/panel4_diffusion.png')

In [ ]:
#@title **Cell 8 — Panel 5: Validation Funnel**
#
# Goal: Ligand funnel from generation to state-selective binders.
# 4 stacked trapezoids, widest at top, narrow at bottom.

fig, ax = plt.subplots(figsize=(10, 10), facecolor=PAL['bg'])
ax.set_xlim(-6, 6)
ax.set_ylim(-1, 11)
ax.set_axis_off()
ax.set_aspect('equal')

stages = [
    {'label': 'Generated\nmolecules',
     'color': '#BFDBFE', 'edge': '#3B82F6',
     'w_top': 5.0, 'w_bot': 4.2, 'n_icons': 14, 'icon_sz': 4},
    {'label': 'Valid OPEN-state\nposes',
     'color': '#FDE68A', 'edge': '#F59E0B',
     'w_top': 4.2, 'w_bot': 3.0, 'n_icons': 8, 'icon_sz': 5},
    {'label': 'Selective \u0394\u0394G\n(open > closed)',
     'color': '#C4B5FD', 'edge': '#7C3AED',
     'w_top': 3.0, 'w_bot': 1.8, 'n_icons': 4, 'icon_sz': 6},
    {'label': 'State-selective\nbinders',
     'color': '#A7F3D0', 'edge': '#059669',
     'w_top': 1.8, 'w_bot': 1.0, 'n_icons': 2, 'icon_sz': 8},
]

y_tops = [8.5, 6.0, 3.5, 1.0]
h = 2.0

rng2 = np.random.default_rng(10)
for stage, yt in zip(stages, y_tops):
    wt, wb = stage['w_top'], stage['w_bot']
    yb = yt - h
    trap = Polygon(
        [[-wt, yt], [wt, yt], [wb, yb], [-wb, yb]],
        closed=True, facecolor=stage['color'],
        edgecolor=stage['edge'], linewidth=2.5, alpha=0.85)
    ax.add_patch(trap)
    ax.text(0, (yt + yb) / 2, stage['label'], ha='center', va='center',
            fontsize=12, fontweight='bold', color=PAL['text'])
    # small molecule icons
    for _ in range(stage['n_icons']):
        ix = rng2.uniform(-wt * 0.55, wt * 0.55)
        iy = rng2.uniform(yb + 0.3, yt - 0.3)
        ax.plot(ix, iy, 'o', color=stage['edge'],
                markersize=stage['icon_sz'], alpha=0.45)

# Side arrow
ax.annotate('', xy=(-5.5, 0.5), xytext=(-5.5, 10.0),
            arrowprops=dict(arrowstyle='->', color=PAL['text'], lw=3))
ax.text(-5.5, 5.5, 'In silico +\nexperimental\nfiltering',
        ha='center', va='center', fontsize=10, fontweight='bold',
        color=PAL['text'], rotation=90)

ax.set_title('VALIDATION FUNNEL', fontsize=16, fontweight='bold',
             color=PAL['text'], pad=15)
fig.text(0.5, 0.01,
         'Multistage filtering yields experimentally verified, '
         'state-selective binders\nthat stabilize the OPEN pocket.',
         ha='center', fontsize=11, color=PAL['text'], style='italic')

plt.savefig('panels/panel5_funnel.png', dpi=DPI, bbox_inches='tight',
            facecolor=PAL['bg'])
plt.show()
print('Panel 5 saved → panels/panel5_funnel.png')

In [ ]:
#@title **Cell 9 — Composite: Assemble All Panels**
#
# Layout:
#   Row 1: Panel 1 (A) | Panel 2 (B)   — Problem
#   Row 2: Panel 3 (C) | Panel 4 (D)   — Method
#   Row 3: Panel 5 (E) spanning full width — Outcome

from PIL import Image, ImageDraw, ImageFont
import glob, os

panel_files = sorted(glob.glob('panels/panel*.png'))
panels = {os.path.basename(f): Image.open(f) for f in panel_files}
print('Loaded panels:', list(panels.keys()))

target_w = 4000
margin = 50
col_w = (target_w - 3 * margin) // 2

def resize_w(img, w):
    ratio = w / img.width
    return img.resize((w, int(img.height * ratio)), Image.LANCZOS)

p1 = resize_w(panels['panel1_idp_ensemble.png'], col_w)
p2 = resize_w(panels['panel2_static_vs_dynamic.png'], col_w)
p3 = resize_w(panels['panel3_pocket_states.png'], col_w)
p4 = resize_w(panels['panel4_diffusion.png'], col_w)
p5 = resize_w(panels['panel5_funnel.png'], target_w - 2 * margin)

row1_h = max(p1.height, p2.height)
row2_h = max(p3.height, p4.height)
row3_h = p5.height
title_h = 100
total_h = title_h + margin * 4 + row1_h + row2_h + row3_h

composite = Image.new('RGB', (target_w, total_h), 'white')
draw = ImageDraw.Draw(composite)

# Fonts
try:
    font_title = ImageFont.truetype(
        '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 52)
    font_label = ImageFont.truetype(
        '/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 36)
    font_row = ImageFont.truetype(
        '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 28)
except Exception:
    font_title = ImageFont.load_default()
    font_label = font_title
    font_row = font_title

# Title
draw.text((target_w // 2, 35),
          'Visual Abstract: State-Aware IDP Ensemble Modeling',
          fill='#1a1a2e', font=font_title, anchor='mt')

y = title_h + margin

# Row 1 label
draw.text((margin, y - 30), 'PROBLEM', fill='#6B7280', font=font_row)

composite.paste(p1, (margin, y))
composite.paste(p2, (margin + col_w + margin, y))
draw.text((margin + 8, y + 5), 'A', fill='#1a1a2e', font=font_label)
draw.text((margin + col_w + margin + 8, y + 5), 'B',
          fill='#1a1a2e', font=font_label)
y += row1_h + margin

# Row 2 label
draw.text((margin, y - 30), 'METHOD', fill='#6B7280', font=font_row)

composite.paste(p3, (margin, y))
composite.paste(p4, (margin + col_w + margin, y))
draw.text((margin + 8, y + 5), 'C', fill='#1a1a2e', font=font_label)
draw.text((margin + col_w + margin + 8, y + 5), 'D',
          fill='#1a1a2e', font=font_label)
y += row2_h + margin

# Row 3 label
draw.text((margin, y - 30), 'OUTCOME', fill='#6B7280', font=font_row)

composite.paste(p5, (margin, y))
draw.text((margin + 8, y + 5), 'E', fill='#1a1a2e', font=font_label)

composite.save('visual_abstract_composite.png', dpi=(300, 300))
print(f'\nComposite saved: visual_abstract_composite.png')
print(f'Dimensions: {composite.size[0]} x {composite.size[1]} px')

# Display a preview (down-scaled)
preview = composite.resize((composite.width // 3, composite.height // 3),
                            Image.LANCZOS)
display(preview)

In [ ]:
#@title **Cell 10 — Download All Files**
#
# Run this cell to zip everything and download from Colab.

import shutil
shutil.make_archive('visual_abstract_panels', 'zip', '.', 'panels')

# Also include the composite
import zipfile
with zipfile.ZipFile('visual_abstract_all.zip', 'w') as zf:
    for f in glob.glob('panels/*.png'):
        zf.write(f)
    zf.write('visual_abstract_composite.png')

print('Created visual_abstract_all.zip')
print('Contents:')
with zipfile.ZipFile('visual_abstract_all.zip', 'r') as zf:
    for name in zf.namelist():
        print(f'  {name}')

# Auto-download in Colab
try:
    from google.colab import files
    files.download('visual_abstract_all.zip')
except ImportError:
    print('Not in Colab — find the zip in your working directory.')